# sensitivity_extend (3).ipynb

Sensitivity, intensity, probe-stability, demographic, cosine-distance, and directional-alignment figures.

Set the cached-result paths and output directory, then run the figure sections you need.

Use the setup steps in the repository README before running this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Change only these paths if the cached files were moved.
PROBING_ROOT = Path("/content/drive/MyDrive/probing")
RESULT_FILES = {
    "QWEN": PROBING_ROOT / "full_probe/albe/shift_results.csv",
    "FLAMINGO": PROBING_ROOT / "full_probe_flamingo/abla/shift_results.csv",
}
CONDITION_RESULT_FILES = {
    "QWEN": PROBING_ROOT / "full_probe/albe/condition_accuracy_shift.csv",
    "FLAMINGO": PROBING_ROOT / "full_probe_flamingo/abla/condition_accuracy_shift.csv",
}
MERGED_RESULT_FILES = {
    "QWEN": PROBING_ROOT / "full_probe/albe/merged_results.csv",
    "FLAMINGO": PROBING_ROOT / "full_probe_flamingo/abla/merged_results.csv",
}
PROBE_INDEX_FILES = {
    "QWEN": Path("/content/drive/MyDrive/probe/probe_representations_2/probe_rep_index.csv"),
    "FLAMINGO": Path("/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_rep_index.csv"),
}
PROBE_RESULT_FILES = {
    "QWEN": {
        "all_layers": Path("/content/drive/MyDrive/probe/probe_representations_2/probe_results_all_layers.csv"),
        "seed_1": Path("/content/drive/MyDrive/probe/probe_representations_2/probe_results_seed_1.csv"),
        "seed_2": Path("/content/drive/MyDrive/probe/probe_representations_2/probe_results_seed_2.csv"),
    },
    "FLAMINGO": {
        "all_layers": Path("/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_results_all_layers.csv"),
        "seed_1": Path("/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_results_seed_1.csv"),
        "seed_2": Path("/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_results_seed_2.csv"),
    },
}
COSINE_RESULT_FILES = {
    "QWEN": PROBING_ROOT / "full_probe/albe/cosine_shift_results.csv",
    "FLAMINGO": PROBING_ROOT / "full_probe_flamingo/abla/cosine_shift_results.csv",
}
DEMOGRAPHIC_CSV = PROBING_ROOT / "VideoDemographics.csv"
OUTPUT_DIR = PROBING_ROOT / "paper_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The original figures were drawn on a 42-inch-wide canvas and later reduced
# to paper width. These explicit values enlarge only their typography.
PANEL_TITLE_FONT = 26
AXIS_LABEL_FONT = 23
TICK_FONT = 23
LEGEND_FONT = 22

print("Output directory:", OUTPUT_DIR)

## Load the cached sensitivity results


In [ ]:
METRIC_COLUMNS = [
    "target_conf_shift",
    "conf_shift",
    "entropy_shift",
    "acc_shift",
]
REQUIRED_COLUMNS = {"task", "layer", "intensity_level", *METRIC_COLUMNS}

def load_shift_results(model, path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{model}: cached file not found: {path}\n"
            "Run the final LO/HI sensitivity-analysis cell in sensitive.ipynb once, "
            "or update RESULT_FILES above."
        )

    frame = pd.read_csv(path)
    missing = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"{model}: {path} is missing columns: {missing}")

    frame = frame.copy()
    frame["task"] = frame["task"].astype(str).str.strip().str.lower()
    frame["layer"] = frame["layer"].astype(str).str.strip().str.lower()
    frame["intensity_level"] = (
        frame["intensity_level"].astype(str).str.strip().str.upper()
    )

    for column in METRIC_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
        if frame[column].notna().sum() == 0:
            raise ValueError(f"{model}: {column} contains no numeric values")

    missing_tasks = {"style", "intensity"} - set(frame["task"])
    if missing_tasks:
        raise ValueError(f"{model}: missing task rows: {sorted(missing_tasks)}")

    intensity_levels = set(
        frame.loc[frame["task"] == "intensity", "intensity_level"]
    )
    missing_levels = {"LO", "HI"} - intensity_levels
    if missing_levels:
        raise ValueError(f"{model}: missing intensity levels: {sorted(missing_levels)}")

    return frame

RESULTS = {
    model: load_shift_results(model, path)
    for model, path in RESULT_FILES.items()
}

inventory = pd.DataFrame([
    {
        "Model": model,
        "Rows": len(frame),
        "Layers": frame["layer"].nunique(),
        "Style rows": int((frame["task"] == "style").sum()),
        "Intensity rows": int((frame["task"] == "intensity").sum()),
    }
    for model, frame in RESULTS.items()
])
display(inventory)

## Shared helpers


In [ ]:
METRICS = [
    ("target_conf_shift", "Target Emotion Confidence Shift"),
    ("conf_shift", "Max Confidence Shift"),
    ("entropy_shift", "Entropy Shift"),
    ("acc_shift", "Emotion Shift Rate"),
]

def natural_sort_key(layer):
    layer = str(layer)
    numbers = re.findall(r"\d+", layer)
    if numbers:
        return layer.split(numbers[-1])[0], int(numbers[-1])
    return layer, 0

def layer_summary(frame, metric):
    stats = frame.groupby("layer", observed=True)[metric].agg(["mean", "std"]).reset_index()
    stats["sort_key"] = stats["layer"].apply(natural_sort_key)
    stats = stats.sort_values("sort_key")
    stats["std"] = stats["std"].fillna(0.0)
    return stats

def enlarge_fonts(ax):
    ax.tick_params(axis="both", labelsize=TICK_FONT)
    legend = ax.get_legend()
    if legend is not None:
        for text in legend.get_texts():
            text.set_fontsize(LEGEND_FONT)

def save_figure(fig, stem):
    paths = {}
    for extension in ("pdf", "svg"):
        path = OUTPUT_DIR / f"{stem}.{extension}"
        fig.savefig(path, bbox_inches="tight")
        paths[extension] = path
    png_path = OUTPUT_DIR / f"{stem}.png"
    fig.savefig(png_path, dpi=200, bbox_inches="tight")
    paths["png"] = png_path
    return paths

## Figure 3: style versus intensity


In [ ]:
def plot_style_vs_intensity(all_shift_df):
    fig, axes = plt.subplots(2, 8, figsize=(42, 10))

    for row_idx, (model_name, shift_df) in enumerate(all_shift_df.items()):
        audio_df = shift_df[~shift_df["layer"].str.startswith("lm_")]
        llm_df = shift_df[shift_df["layer"].str.startswith("lm_")]
        groups = [("Audio + Projector", audio_df), ("LLM", llm_df)]
        task_types = sorted(shift_df["task"].unique())
        colors = ["gold", "blue"]

        for group_idx, (group_name, group_df) in enumerate(groups):
            for metric_idx, (metric, title) in enumerate(METRICS):
                col_idx = group_idx * 4 + metric_idx
                ax = axes[row_idx, col_idx]

                for task_idx, task_name in enumerate(task_types):
                    task_subset = group_df[group_df["task"] == task_name]
                    stats = layer_summary(task_subset, metric)
                    x = np.arange(len(stats))
                    mean = stats["mean"].to_numpy(dtype=float)
                    std = stats["std"].to_numpy(dtype=float)
                    color = colors[task_idx % len(colors)]

                    ax.plot(x, mean, label=task_name, color=color)
                    ax.fill_between(x, mean - std, mean + std, alpha=0.20, color=color)

                ax.axhline(0, linestyle="--", linewidth=1, color="black")
                ax.set_title(
                    f"{model_name}\n{group_name}\n{title}",
                    fontsize=PANEL_TITLE_FONT,
                )
                ax.set_xlabel("Layer", fontsize=AXIS_LABEL_FONT)
                if col_idx == 0:
                    ax.set_ylabel("Shift", fontsize=AXIS_LABEL_FONT)
                ax.grid(True, alpha=0.3)
                # Show the legend only on the first/top-left subplot.
                if row_idx == 0 and col_idx == 0:
                    ax.legend(
                        loc="upper right",
                        fontsize=LEGEND_FONT,
                    )

                enlarge_fonts(ax)
    plt.tight_layout()
    paths = save_figure(fig, "figure3_sensitivity_style_vs_intensity")
    return fig, axes, paths

fig3, axes3, figure3_paths = plot_style_vs_intensity(RESULTS)
plt.show()
figure3_paths

## Figure 4: low versus high intensity


In [ ]:
def plot_low_vs_high(all_shift_df):
    fig, axes = plt.subplots(2, 8, figsize=(42, 10))

    for row_idx, (model_name, shift_df) in enumerate(all_shift_df.items()):
        intensity_df = shift_df[
            (shift_df["task"] == "intensity")
            & (shift_df["intensity_level"].isin(["LO", "HI"]))
        ]
        audio_df = intensity_df[~intensity_df["layer"].str.startswith("lm_")]
        llm_df = intensity_df[intensity_df["layer"].str.startswith("lm_")]
        groups = [("Audio + Projector", audio_df), ("LLM", llm_df)]
        intensity_colors = {"LO": "blue", "HI": "red"}

        for group_idx, (group_name, group_df) in enumerate(groups):
            for metric_idx, (metric, title) in enumerate(METRICS):
                col_idx = group_idx * 4 + metric_idx
                ax = axes[row_idx, col_idx]

                for intensity in ["LO", "HI"]:
                    subset = group_df[group_df["intensity_level"] == intensity]
                    if subset.empty:
                        continue
                    stats = layer_summary(subset, metric)
                    x = np.arange(len(stats))
                    mean = stats["mean"].to_numpy(dtype=float)
                    std = stats["std"].to_numpy(dtype=float)

                    ax.plot(x, mean, label=intensity, color=intensity_colors[intensity])
                    ax.fill_between(
                        x, mean - std, mean + std,
                        alpha=0.20, color=intensity_colors[intensity],
                    )

                ax.axhline(0, linestyle="--", linewidth=1, color="black")
                ax.set_title(
                    f"{model_name}\n{group_name}\n{title}",
                    fontsize=PANEL_TITLE_FONT,
                )
                ax.set_xlabel("Layer", fontsize=AXIS_LABEL_FONT)
                if col_idx == 0:
                    ax.set_ylabel("Shift", fontsize=AXIS_LABEL_FONT)
                ax.grid(True, alpha=0.3)
                # Show the legend only on the first/top-left subplot.
                if row_idx == 0 and col_idx == 0:
                    ax.legend(
                        loc="upper right",
                        fontsize=LEGEND_FONT,
                    )

                enlarge_fonts(ax)
    plt.tight_layout()
    paths = save_figure(fig, "figure4_sensitivity_low_vs_high")
    return fig, axes, paths

fig4, axes4, figure4_paths = plot_low_vs_high(RESULTS)
plt.show()
figure4_paths

## Intensity-task accuracy


In [ ]:
EMOTION_MAP = {
    "ang": "ANG", "anger": "ANG", "angry": "ANG",
    "dis": "DIS", "disgust": "DIS",
    "fea": "FEA", "fear": "FEA", "fearful": "FEA",
    "hap": "HAP", "happy": "HAP", "happiness": "HAP",
    "sad": "SAD", "sadness": "SAD",
    "neu": "NEU", "neutral": "NEU",
    "sur": "SUR", "surprise": "SUR", "surprised": "SUR",
}
VALID_EMOTIONS = {"ANG", "DIS", "FEA", "HAP", "SAD", "NEU", "SUR"}

def basename_without_extension(value):
    filename = re.split(r"[\\/]", str(value))[-1]
    return Path(filename).stem

def extract_true_emotion(sample_id):
    try:
        return basename_without_extension(sample_id).split("_")[-1].upper()
    except Exception:
        return None

def extract_intensity(path):
    try:
        level = basename_without_extension(path).split("_")[-1].upper()
        return level if level in {"LO", "MD", "HI"} else None
    except Exception:
        return None

def normalize_behavior_prediction(text):
    if pd.isna(text):
        return None
    text = str(text).lower().strip()
    if re.search(r"\bneutral\b", text):
        return "NEU"
    for key, value in EMOTION_MAP.items():
        if re.search(rf"\b{re.escape(key)}\b", text):
            return value
    return None

intensity_rows = []
behavior_required = {"task", "id", "manipulated_path", "raw_response", "neutral_raw"}

for model_name, csv_path in MERGED_RESULT_FILES.items():
    if not csv_path.exists():
        raise FileNotFoundError(f"{model_name}: merged result file not found: {csv_path}")
    frame = pd.read_csv(csv_path)
    missing = sorted(behavior_required - set(frame.columns))
    if missing:
        raise ValueError(f"{model_name}: missing behavioral columns: {missing}")

    frame = frame.loc[frame["task"].astype(str).str.lower() == "intensity"].copy()
    frame["true_emotion"] = frame["id"].apply(extract_true_emotion)
    frame["intensity_level"] = frame["manipulated_path"].apply(extract_intensity)
    frame["manip_pred"] = frame["raw_response"].apply(normalize_behavior_prediction)
    frame["baseline_pred"] = frame["neutral_raw"].apply(normalize_behavior_prediction)
    frame = frame.loc[
        frame["manip_pred"].isin(VALID_EMOTIONS)
        & frame["baseline_pred"].isin(VALID_EMOTIONS)
    ].copy()
    frame["manip_correct"] = (frame["manip_pred"] == frame["true_emotion"]).astype(int)
    frame["baseline_correct"] = (frame["baseline_pred"] == frame["true_emotion"]).astype(int)

    for intensity in ["LO", "MD", "HI"]:
        if intensity == "MD":
            accuracy = frame["baseline_correct"].mean()
        else:
            accuracy = frame.loc[frame["intensity_level"] == intensity, "manip_correct"].mean()
        intensity_rows.append({"model": model_name, "intensity": intensity, "accuracy": accuracy})

intensity_plot_df = pd.DataFrame(intensity_rows)
display(intensity_plot_df)

sns.set_style("whitegrid")
fig_intensity, ax = plt.subplots(figsize=(7, 5))
sns.barplot(
    data=intensity_plot_df, x="intensity", y="accuracy", hue="model",
    order=["LO", "MD", "HI"], ax=ax,
)
ax.set_title("Intensity Task Accuracy", fontsize=PANEL_TITLE_FONT)
ax.set_ylabel("Accuracy", fontsize=AXIS_LABEL_FONT)
ax.set_xlabel("Intensity", fontsize=AXIS_LABEL_FONT)
ax.set_ylim(0, 1)
ax.legend(title="model", fontsize=LEGEND_FONT, title_fontsize=LEGEND_FONT)
enlarge_fonts(ax)
fig_intensity.tight_layout()
intensity_accuracy_paths = save_figure(fig_intensity, "figure_intensity_task_accuracy")
plt.show()
intensity_accuracy_paths

## Cue-condition layer-wise accuracy shifts


In [ ]:
condition_results = {}
condition_required = {"condition", "layer", "acc_shift"}
for model_name, csv_path in CONDITION_RESULT_FILES.items():
    if not csv_path.exists():
        raise FileNotFoundError(
            f"{model_name}: condition result file not found: {csv_path}\n"
            "Run the condition-specific sensitivity cell in sensitive.ipynb once."
        )
    frame = pd.read_csv(csv_path)
    missing = sorted(condition_required - set(frame.columns))
    if missing:
        raise ValueError(f"{model_name}: missing condition columns: {missing}")
    frame = frame.copy()
    frame["condition"] = frame["condition"].astype(str).str.strip().str.lower()
    frame["layer"] = frame["layer"].astype(str).str.strip().str.lower()
    frame["acc_shift"] = pd.to_numeric(frame["acc_shift"], errors="coerce")
    condition_results[model_name] = frame

def plot_condition_accuracy_shift(all_results):
    conditions = sorted(set().union(*(set(frame["condition"]) for frame in all_results.values())))
    cmap = plt.get_cmap("tab10", len(conditions))
    fig, axes = plt.subplots(2, 2, figsize=(20, 10))
    axes = axes.flatten()

    for plot_idx, (model_name, shift_df) in enumerate(all_results.items()):
        audio_df = shift_df[~shift_df["layer"].str.startswith("lm_")]
        llm_df = shift_df[shift_df["layer"].str.startswith("lm_")]
        groups = [("Audio + Projector", audio_df), ("LLM", llm_df)]

        for group_idx, (group_name, group_df) in enumerate(groups):
            ax = axes[plot_idx * 2 + group_idx]
            layers = sorted(group_df["layer"].dropna().unique(), key=natural_sort_key)
            x = np.arange(len(layers))

            for condition_idx, condition in enumerate(conditions):
                subset = group_df[group_df["condition"] == condition]
                if subset.empty:
                    continue
                stats = (
                    subset.groupby("layer", observed=True)["acc_shift"]
                    .agg(["mean", "std"]).reindex(layers)
                )
                stats["std"] = stats["std"].fillna(0.0)
                mean = stats["mean"].to_numpy(dtype=float)
                std = stats["std"].to_numpy(dtype=float)
                color = cmap(condition_idx)
                ax.plot(x, mean, label=condition, color=color, linewidth=2)
                ax.fill_between(x, mean - std, mean + std, alpha=0.20, color=color)

            ax.axhline(0, linestyle="--", color="black", linewidth=1)
            ax.set_xticks(x)
            ax.set_xticklabels(layers, rotation=45, ha="right", fontsize=TICK_FONT)
            ax.set_title(
                f"{model_name}\n{group_name}\nAccuracy Shift",
                fontsize=PANEL_TITLE_FONT,
            )
            ax.set_xlabel("Layer", fontsize=AXIS_LABEL_FONT)
            ax.set_ylabel("Accuracy Shift", fontsize=AXIS_LABEL_FONT)
            ax.grid(True, alpha=0.3)
            # Show one shared legend only in the QWEN LLM panel.
            if model_name.upper() == "QWEN" and group_name == "LLM":
                ax.legend(loc="best", fontsize=LEGEND_FONT)
            else:
                legend = ax.get_legend()
                if legend is not None:
                    legend.remove()

            enlarge_fonts(ax)

    fig.tight_layout()
    paths = save_figure(fig, "appendix_cue_condition_accuracy_shift")
    return fig, axes, paths

fig_conditions, axes_conditions, condition_shift_paths = plot_condition_accuracy_shift(condition_results)
plt.show()
condition_shift_paths

## CREMA-D demographic distribution and accuracy


In [ ]:
DEMOGRAPHIC_EMOTION_MAP = {
    "ang": "ANG", "anger": "ANG", "angry": "ANG",
    "dis": "DIS", "disgust": "DIS", "disgusted": "DIS",
    "fea": "FEA", "fear": "FEA", "fearful": "FEA",
    "hap": "HAP", "happy": "HAP",
    "sad": "SAD", "sadness": "SAD",
    "neu": "NEU", "neutral": "NEU",
    "sur": "SUR", "surprise": "SUR", "surprised": "SUR",
}

def normalize_probe_prediction(text):
    if pd.isna(text):
        return None
    text = str(text).lower().strip()
    for key, value in DEMOGRAPHIC_EMOTION_MAP.items():
        if re.search(rf"\b{re.escape(key)}\b", text):
            return value
    return None

def age_bucket(age):
    try:
        age = int(age)
    except (TypeError, ValueError):
        return None
    if age < 35:
        return "Young"
    if age < 50:
        return "Middle"
    return "Older"

if not DEMOGRAPHIC_CSV.exists():
    raise FileNotFoundError(f"Demographic metadata not found: {DEMOGRAPHIC_CSV}")
demo_df = pd.read_csv(DEMOGRAPHIC_CSV)
if "ActorID" not in demo_df.columns:
    raise ValueError(f"{DEMOGRAPHIC_CSV} is missing ActorID")
demo_df["ActorID"] = demo_df["ActorID"].astype(str)

all_demographic_results = []
all_distribution_rows = []
probe_required = {"emotion", "llm_output", "actor"}
demographic_names = ["Sex", "Race", "Ethnicity", "AgeGroup"]

for model_name, csv_path in PROBE_INDEX_FILES.items():
    if not csv_path.exists():
        raise FileNotFoundError(f"{model_name}: probe index not found: {csv_path}")
    frame = pd.read_csv(csv_path)
    missing = sorted(probe_required - set(frame.columns))
    if missing:
        raise ValueError(f"{model_name}: missing probe-index columns: {missing}")

    frame["true_emotion"] = frame["emotion"].astype(str).str.upper()
    frame["pred_emotion"] = frame["llm_output"].apply(normalize_probe_prediction)
    frame = frame.loc[frame["pred_emotion"].isin(VALID_EMOTIONS)].copy()
    frame["correct"] = (frame["pred_emotion"] == frame["true_emotion"]).astype(int)
    frame["ActorID"] = frame["actor"].astype(str)
    frame = frame.merge(demo_df, on="ActorID", how="left")
    frame = frame.loc[frame["Race"].notna() & (frame["Race"] != "Unknown")].copy()
    frame["AgeGroup"] = frame["Age"].apply(age_bucket)

    for demographic in demographic_names:
        distribution = frame[demographic].value_counts(dropna=False).reset_index()
        distribution.columns = [demographic, "count"]
        distribution["percentage"] = distribution["count"] / distribution["count"].sum() * 100
        distribution["model"] = model_name
        distribution["demographic_type"] = demographic
        all_distribution_rows.append(distribution)

        accuracy = frame.groupby(demographic, observed=True)["correct"].agg(["mean", "count"]).reset_index()
        accuracy = accuracy.rename(columns={"mean": "accuracy", "count": "samples"})
        accuracy["model"] = model_name
        accuracy["demographic_type"] = demographic
        all_demographic_results.append(accuracy)

demographic_results_df = pd.concat(all_demographic_results, ignore_index=True)
demographic_distribution_df = pd.concat(all_distribution_rows, ignore_index=True)
demographic_results_df.to_csv(OUTPUT_DIR / "demographic_accuracy_results.csv", index=False)
demographic_distribution_df.to_csv(OUTPUT_DIR / "demographic_distribution.csv", index=False)

sns.set_style("whitegrid")
fig_demographic, axes = plt.subplots(2, 4, figsize=(24, 10))

for column, demographic in enumerate(demographic_names):
    ax = axes[0, column]
    subset = demographic_distribution_df[
        demographic_distribution_df["demographic_type"] == demographic
    ]
    sns.barplot(data=subset, x=demographic, y="count", hue="model", ax=ax)
    ax.set_title(f"{demographic} Distribution", fontsize=PANEL_TITLE_FONT)
    ax.set_ylabel("Count", fontsize=AXIS_LABEL_FONT)
    ax.set_xlabel(demographic, fontsize=AXIS_LABEL_FONT)
    ax.tick_params(axis="x", rotation=20, labelsize=TICK_FONT)
    enlarge_fonts(ax)

    ax = axes[1, column]
    subset = demographic_results_df[demographic_results_df["demographic_type"] == demographic]
    sns.barplot(data=subset, x=demographic, y="accuracy", hue="model", ax=ax)
    ax.set_title(f"{demographic} Accuracy", fontsize=PANEL_TITLE_FONT)
    ax.set_ylabel("Accuracy", fontsize=AXIS_LABEL_FONT)
    ax.set_xlabel(demographic, fontsize=AXIS_LABEL_FONT)
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=20, labelsize=TICK_FONT)
    enlarge_fonts(ax)

for row in range(2):
    for column in range(4):
        if column != 3 and axes[row, column].get_legend() is not None:
            axes[row, column].get_legend().remove()

fig_demographic.tight_layout()
demographic_paths = save_figure(fig_demographic, "figure_demographic_accuracy")
plt.show()
demographic_paths

## Probe accuracy across three random seeds


In [ ]:
CHANCE_LEVEL = 1 / 7

def load_probe_seed_result(path, seed_name):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Probe result not found: {path}")
    frame = pd.read_csv(path)

    accuracy_column = next(
        (column for column in ["test_acc", "accuracy", "acc", "probe_acc"] if column in frame.columns),
        None,
    )
    layer_column = next(
        (column for column in ["layer", "layer_name"] if column in frame.columns),
        None,
    )
    if accuracy_column is None:
        raise ValueError(f"No accuracy column found in {path}")
    if layer_column is None:
        raise ValueError(f"No layer column found in {path}")

    frame = frame[[layer_column, accuracy_column]].copy()
    frame.columns = ["layer", "accuracy"]
    frame["layer"] = frame["layer"].astype(str)
    frame["accuracy"] = pd.to_numeric(frame["accuracy"], errors="raise")
    frame["seed"] = seed_name
    return frame

all_probe_results = {}
for model_name, seed_paths in PROBE_RESULT_FILES.items():
    seed_frames = [
        load_probe_seed_result(path, seed_name)
        for seed_name, path in seed_paths.items()
    ]
    merged = pd.concat(seed_frames, ignore_index=True)
    merged["sort_key"] = merged["layer"].apply(natural_sort_key)
    all_probe_results[model_name] = merged.sort_values("sort_key")

fig_probe_accuracy, axes_probe_accuracy = plt.subplots(2, 2, figsize=(22, 12))
axes_probe_accuracy = axes_probe_accuracy.flatten()

for plot_idx, (model_name, frame) in enumerate(all_probe_results.items()):
    audio_frame = frame[~frame["layer"].str.startswith("lm_")].copy()
    llm_frame = frame[frame["layer"].str.startswith("lm_")].copy()

    for group_idx, (group_name, group_frame) in enumerate([
        ("Audio + Projector", audio_frame),
        ("LLM", llm_frame),
    ]):
        ax = axes_probe_accuracy[plot_idx * 2 + group_idx]
        stats = group_frame.groupby("layer", observed=True)["accuracy"].agg(["mean", "std"]).reset_index()
        stats["sort_key"] = stats["layer"].apply(natural_sort_key)
        stats = stats.sort_values("sort_key")
        x = np.arange(len(stats))
        mean = stats["mean"].to_numpy(dtype=float)
        std = stats["std"].fillna(0.0).to_numpy(dtype=float)

        ax.plot(x, mean, linewidth=2)
        ax.fill_between(x, mean - std, mean + std, alpha=0.20)
        ax.axhline(
            CHANCE_LEVEL, linestyle="--", color="red", linewidth=1.5, label="Chance"
        )
        ax.set_title(
            f"{model_name}\n{group_name}\nProbe Accuracy Across Layers",
            fontsize=PANEL_TITLE_FONT,
        )
        ax.set_xlabel("Layer", fontsize=AXIS_LABEL_FONT)
        ax.set_ylabel("Probe Accuracy", fontsize=AXIS_LABEL_FONT)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=LEGEND_FONT)
        enlarge_fonts(ax)

fig_probe_accuracy.tight_layout()
probe_accuracy_paths = save_figure(fig_probe_accuracy, "appendix_probe_accuracy_three_seeds")
plt.show()

probe_summary_rows = []
for model_name, frame in all_probe_results.items():
    for group_name, group_frame in [
        ("Audio + Projector", frame[~frame["layer"].str.startswith("lm_")]),
        ("LLM", frame[frame["layer"].str.startswith("lm_")]),
    ]:
        stats = group_frame.groupby("layer", observed=True)["accuracy"].agg(["mean", "std"]).reset_index()
        peak = stats.loc[stats["mean"].idxmax()]
        probe_summary_rows.append({
            "model": model_name,
            "group": group_name,
            "peak_layer": peak["layer"],
            "peak_accuracy": round(float(peak["mean"]), 4),
            "std_at_peak": round(float(peak["std"]), 4),
        })

probe_summary_df = pd.DataFrame(probe_summary_rows)
probe_summary_df.to_csv(OUTPUT_DIR / "probe_summary.csv", index=False)
display(probe_summary_df)
probe_accuracy_paths

## Baseline-to-manipulated cosine distance


In [ ]:
cosine_results = {}
cosine_required = {"task", "layer", "cosine_distance"}

for model_name, csv_path in COSINE_RESULT_FILES.items():
    if not csv_path.exists():
        raise FileNotFoundError(
            f"{model_name}: cached cosine results not found: {csv_path}\n"
            "Run the PURE COSINE REPRESENTATION SHIFT ANALYSIS cell in sensitive.ipynb once."
        )
    frame = pd.read_csv(csv_path)
    missing = sorted(cosine_required - set(frame.columns))
    if missing:
        raise ValueError(f"{model_name}: missing cosine-result columns: {missing}")
    frame = frame.copy()
    frame["layer"] = frame["layer"].astype(str)
    frame["cosine_distance"] = pd.to_numeric(frame["cosine_distance"], errors="raise")
    cosine_results[model_name] = frame

def plot_cosine_results(all_shift_df):
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))

    for row_idx, (model_name, shift_df) in enumerate(all_shift_df.items()):
        audio_df = shift_df[~shift_df["layer"].str.startswith("lm_")]
        llm_df = shift_df[shift_df["layer"].str.startswith("lm_")]

        for col_idx, (group_name, group_df) in enumerate([
            ("Audio + Projector", audio_df),
            ("LLM", llm_df),
        ]):
            ax = axes[row_idx, col_idx]
            task_types = sorted(group_df["task"].dropna().unique())

            for task_name in task_types:
                subset = group_df[group_df["task"] == task_name]
                stats = subset.groupby("layer", observed=True)["cosine_distance"].agg(["mean", "std"]).reset_index()
                stats["sort_key"] = stats["layer"].apply(natural_sort_key)
                stats = stats.sort_values("sort_key")
                x = np.arange(len(stats))
                mean = stats["mean"].to_numpy(dtype=float)
                std = stats["std"].fillna(0.0).to_numpy(dtype=float)

                ax.plot(x, mean, label=task_name)
                ax.fill_between(x, mean - std, mean + std, alpha=0.20)

            ax.set_title(
                f"{model_name}\n{group_name}\nCosine Distance",
                fontsize=PANEL_TITLE_FONT,
            )
            ax.set_xlabel("Layer", fontsize=AXIS_LABEL_FONT)
            ax.set_ylabel("Cosine Distance", fontsize=AXIS_LABEL_FONT)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=LEGEND_FONT)
            enlarge_fonts(ax)

    fig.tight_layout()
    paths = save_figure(fig, "appendix_cosine_representation_shift")
    return fig, axes, paths

fig_cosine, axes_cosine, cosine_paths = plot_cosine_results(cosine_results)
plt.show()
cosine_paths

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Reuse the completed run.
# If the runtime was restarted, replace this with the saved run directory:
# DIRECTION_RUN_DIR = Path("/content/drive/MyDrive/.../directional_alignment_...")

DIRECTION_RUN_DIR = Path("/content/drive/MyDrive/subprobe/geometry_unified/run_20260913T133917Z_c79a33ce")

summary_path = (
    DIRECTION_RUN_DIR
    / "cremad_directional_alignment_summary.csv"
)

if not summary_path.exists():
    raise FileNotFoundError(
        f"Saved directional summary not found: {summary_path}"
    )

directional_summary_saved = pd.read_csv(summary_path)

TRANSITIONS_PLOT = ["MD_TO_HI", "MD_TO_LO"]
COARSE_ORDER_PLOT = ["ANG", "DIS", "FEA", "HAP", "SAD"]


def plot_cremad_directional_alignment_large(summary, destination):
    colors = {
        "MD_TO_HI": "#D55E00",
        "MD_TO_LO": "#0072B2",
    }
    transition_labels = {
        "MD_TO_HI": "MD→HI",
        "MD_TO_LO": "MD→LO",
    }
    coarse_labels = {
        "ANG": "Anger",
        "DIS": "Disgust",
        "FEA": "Fear",
        "HAP": "Happiness",
        "SAD": "Sadness",
    }

    # Preserve the model ordering already stored in the results.
    models = summary["Model"].drop_duplicates().tolist()

    # Only typography is enlarged.
    style = {
        "font.size": 13,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 13,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }

    with plt.rc_context(style):
        fig, axes = plt.subplots(
            1,
            len(models),
            figsize=(4.8 * len(models), 3.5),
            sharey=True,
            squeeze=False,
        )

        x = np.arange(len(COARSE_ORDER_PLOT))
        width = 0.36

        for ax, model in zip(axes[0], models):
            matrix = (
                summary[summary["Model"].eq(model)]
                .pivot(
                    index="Coarse",
                    columns="Transition",
                    values="mean_subtle_alignment",
                )
                .reindex(
                    index=COARSE_ORDER_PLOT,
                    columns=TRANSITIONS_PLOT,
                )
            )

            if matrix.isna().any().any():
                raise ValueError(
                    f"Incomplete figure matrix for {model}"
                )

            for offset, transition in zip(
                [-width / 2, width / 2],
                TRANSITIONS_PLOT,
            ):
                ax.bar(
                    x + offset,
                    matrix[transition].to_numpy(),
                    width,
                    label=transition_labels[transition],
                    color=colors[transition],
                )

            ax.axhline(
                0,
                color="#444444",
                linewidth=0.9,
            )
            ax.set_xticks(
                x,
                [
                    coarse_labels[emotion]
                    for emotion in COARSE_ORDER_PLOT
                ],
                rotation=25,
                ha="right",
            )
            ax.set_xlabel("CREMA-D coarse emotion")
            ax.set_title(model)
            ax.grid(
                axis="y",
                alpha=0.22,
                linewidth=0.7,
            )
            ax.set_axisbelow(True)

        axes[0, 0].set_ylabel(
            "Cosine alignment with EmoS\n"
            "coarse→subtle directions"
        )

        handles, labels = (
            axes[0, 0].get_legend_handles_labels()
        )
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=2,
            frameon=False,
            bbox_to_anchor=(0.5, 1.06),
        )

        # Reserve space for the larger legend.
        fig.tight_layout(
            rect=[0, 0, 1, 0.90],
            w_pad=1.4,
        )

        destination = Path(destination)
        output_stem = "cremad_directional_alignment_large_font"

        output_paths = {}
        for suffix in ["pdf", "svg", "png"]:
            output_path = destination / f"{output_stem}.{suffix}"
            fig.savefig(
                output_path,
                dpi=300,
                bbox_inches="tight",
                facecolor="white",
            )
            output_paths[suffix] = output_path

        return fig, output_paths


directional_figure_large, directional_figure_paths = (
    plot_cremad_directional_alignment_large(
        directional_summary_saved,
        DIRECTION_RUN_DIR,
    )
)

plt.show()
directional_figure_paths

## Output files
